In [26]:
!rm -r /content/drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [14]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

project_path = "/content/drive/My Drive/Logistic-Softmax-Neural-Nets"
data_path = f"{project_path}/data"

transform = transforms.ToTensor()

train_data_full = datasets.MNIST(
    root=data_path,
    train=True,
    download=True,
    transform=transform
)

test_data_full = datasets.MNIST(
    root=data_path,
    train=False,
    download=True,
    transform=transform
)

# Combine all data
all_data = torch.cat((train_data_full.data, test_data_full.data), dim=0)
all_targets = torch.cat((train_data_full.targets, test_data_full.targets), dim=0)

In [15]:
all_data = all_data.unsqueeze(1).float() / 255.0

train_data, temp_data, train_targets, temp_targets = train_test_split(
    all_data,
    all_targets,
    train_size=0.60,
    stratify=all_targets,
    random_state=42
)

val_data, test_data, val_targets, test_targets = train_test_split(
    temp_data,
    temp_targets,
    train_size=0.50,
    stratify=temp_targets,
    random_state=42
)

In [16]:
BATCH_SIZE = 64

train_dataset = TensorDataset(train_data, train_targets)
val_dataset = TensorDataset(val_data, val_targets)
test_dataset = TensorDataset(test_data, test_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [17]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: [B, 1, 28, 28]
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1, padding=2)
        # Output: [B, 16, 28, 28]

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Output: [B, 16, 14, 14]

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=2)
        # Output: [B, 32, 14, 14]

        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Output: [B, 32, 7, 7]

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # x starts as [B, 1, 28, 28]
        x = self.pool1(F.relu(self.conv1(x))) # Shape: [B, 16, 14, 14]

        x = self.pool2(F.relu(self.conv2(x))) # Shape: [B, 32, 7, 7]

        x = x.view(-1, 32 * 7 * 7) # Shape: [B, 1568]

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [27]:
# Applying He initialization to the model weights
def initialize_weights(module):
    if isinstance(module, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
        nn.init.zeros_(module.bias)

# define device GPU and apply initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)
model.apply(initialize_weights)

CNN(
  (conv1): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [28]:
import torch.optim as optim

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs= 100

train_loss = []
val_loss = []
train_acc = []
val_acc = []

patience = 3
patience_counter = 0
improvement_threshold = 1e-3
best_val_loss = float('inf')
model_path = f"{project_path}/cnn_checkpoint.pth"

In [29]:
for epoch in range(epochs):
    model.train()

    train_loss_epoch = 0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward Pass
        outputs = model(inputs)

        # Calculate Loss
        loss = loss_fn(outputs, labels)

        # Comute Gradients
        loss.backward()
        optimizer.step()

        train_loss_epoch += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy for the epoch
    avg_train_loss = train_loss_epoch / len(train_loader)
    avg_train_acc = 100 * correct_train / total_train

    train_loss.append(avg_train_loss)
    train_acc.append(avg_train_acc)

    # Validation
    model.eval()

    val_loss_epoch = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            val_loss_epoch += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    avg_val_loss = val_loss_epoch / len(val_loader)
    avg_val_acc = 100 * correct_val / total_val

    val_loss.append(avg_val_loss)
    val_acc.append(avg_val_acc)

    # Logging
    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")

    # Early Stopping and saving Checkpoints
    improvement = best_val_loss - avg_val_loss
    if improvement > improvement_threshold:
        best_val_loss = avg_val_loss
        patience_counter = 0

        torch.save(model.state_dict(), model_path)
    else:
        patience_counter += 1
        print(f" Val loss did not improve by {improvement_threshold:.3f}.")
    if patience_counter >= patience:
      print(f"Early stopping after {epoch + 1} epochs")
      break

Epoch [1/100]
  Train Loss: 0.3939 | Train Acc: 88.59%
  Val Loss:   0.1889 | Val Acc:   94.22%
Epoch [2/100]
  Train Loss: 0.1531 | Train Acc: 95.57%
  Val Loss:   0.1478 | Val Acc:   95.67%
Epoch [3/100]
  Train Loss: 0.1101 | Train Acc: 96.61%
  Val Loss:   0.0998 | Val Acc:   96.93%
Epoch [4/100]
  Train Loss: 0.0884 | Train Acc: 97.34%
  Val Loss:   0.1130 | Val Acc:   96.51%
 Val loss did not improve by 0.001.
Epoch [5/100]
  Train Loss: 0.0750 | Train Acc: 97.76%
  Val Loss:   0.0776 | Val Acc:   97.60%
Epoch [6/100]
  Train Loss: 0.0664 | Train Acc: 97.98%
  Val Loss:   0.0733 | Val Acc:   97.85%
Epoch [7/100]
  Train Loss: 0.0598 | Train Acc: 98.13%
  Val Loss:   0.0623 | Val Acc:   98.19%
Epoch [8/100]
  Train Loss: 0.0540 | Train Acc: 98.40%
  Val Loss:   0.0664 | Val Acc:   97.96%
 Val loss did not improve by 0.001.
Epoch [9/100]
  Train Loss: 0.0487 | Train Acc: 98.47%
  Val Loss:   0.0781 | Val Acc:   97.56%
 Val loss did not improve by 0.001.
Epoch [10/100]
  Train Loss:

In [30]:
class FeedForwardNN(nn.Module):
    def __init__ (self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(28*28, 128)
        self.activation = nn.ReLU()
        self.hidden2 = nn.Linear(128, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.hidden1(x))
        x = self.activation(self.hidden2(x))
        x = self.output(x)
        return x

In [31]:
ffn_model_path = f"{project_path}/model_checkpoint.pth"
cnn_model_path = f"{project_path}/cnn_checkpoint.pth"

ffn_model = FeedForwardNN().to(device)
ffn_model.load_state_dict(torch.load(ffn_model_path))

cnn_model = CNN().to(device)
cnn_model.load_state_dict(torch.load(cnn_model_path))

loss_fn = nn.CrossEntropyLoss()

In [32]:
def evaluate_model(model, data_loader, device):
    model.eval()

    total_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    avg_loss = total_loss / len(data_loader)
    avg_acc = 100 * correct_test / total_test

    return avg_loss, avg_acc

ffn_loss, ffn_acc = evaluate_model(ffn_model, test_loader, device)
cnn_loss, cnn_acc = evaluate_model(cnn_model, test_loader, device)

print("\nTest Set Analysis")
print(f"Feedforward NN: Loss: {ffn_loss:.4f} | Accuracy: {ffn_acc:.2f}%")
print(f"Convolutional NN: Loss: {cnn_loss:.4f} | Accuracy: {cnn_acc:.2f}%")


Test Set Analysis
Feedforward NN: Loss: 0.1008 | Accuracy: 97.01%
Convolutional NN: Loss: 0.0469 | Accuracy: 98.64%


## Analyze the Benefit of Spatial Feature Learning

You will see from your test results that **CNN performs significantly better** than the regular feed-forward neural network. This is because **spatial feature learning**.

### The FNN
The `FeedForwardNN` model first flattens the 28×28 image into a 784-pixel vector.  
This **destroys all spatial information**.  
It treats all pixels as independent features.

### The CNN
The CNN model **works directly on the 2D image**.

- **Spatial Learning:**  
  The `nn.Conv2d` layers (kernels or filters) slide over the 2D image, examining **groups of nearby pixels** together.  
  This lets the model understand **local spatial patterns**.

- **Feature Hierarchy:**  
  CNNs build a **hierarchy of features** across layers:
  - **Layer 1 (`conv1`):** Detects simple features such as edges, corners, and small curves.  
  - **Layer 2 (`conv2`):** Combines these features to detect **more complex patterns**, like loops or crosses.  
  - **Final Layers (`fc1`, `fc2`):** Use these learned features to make decisions.
